# 00 · Data audit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/hillstrom-uplift-lab/blob/main/notebooks/00_data_audit.ipynb)

**Goal:** understand the experiment table before analyzing outcomes. We check its shape, data types, missing values, repeated rows, group assignment, and logical consistency. We do not remove identical rows because the source has no customer ID to establish whether they are accidental duplicates.

## Run this notebook in Colab

Each notebook runs independently. The next cell installs the analysis packages, clones the project, and downloads the public dataset. You do not need Kaggle credentials.

In [ ]:
%pip -q install pandas numpy scipy statsmodels scikit-learn plotly


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path("/content/hillstrom-uplift-lab")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ericmavigo/hillstrom-uplift-lab.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "download_data.py"], check=True)
print(f"Project directory: {REPO_DIR}")


## 1. Load the source table

The source contains one row per assigned customer. `segment` is the randomized assignment; visit, purchase, and spend are measured after the email experiment.

In [ ]:
import pandas as pd

DATA_PATH = REPO_DIR / "data" / "raw" / "hillstrom.csv"
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())


## 2. Inspect schema and missingness

**Why:** understand which columns are categorical, numeric, or outcomes before choosing transformations or predictors.

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(),
    "missing": df.isna().sum(),
})
display(profile)
print("Rows with at least one missing field:", int(df.isna().any(axis=1).sum()))


## 3. Check repeated rows and impossible values

Identical rows may belong to different customers with the same observed attributes. Without a customer identifier, treating every identical row as a data error would be an unsupported assumption.

In [ ]:
checks = pd.Series({
    "Exact repeated rows": int(df.duplicated().sum()),
    "Visit/conversion values outside 0 or 1": int((~df.visit.isin([0, 1])).sum() + (~df.conversion.isin([0, 1])).sum()),
    "Negative spend": int((df.spend < 0).sum()),
    "Purchase without visit": int(((df.conversion == 1) & (df.visit == 0)).sum()),
    "Positive spend without purchase": int(((df.spend > 0) & (df.conversion == 0)).sum()),
})
display(checks.to_frame("count"))


## 4. Check randomized group sizes and outcomes

The groups should be close in size because customers were randomly assigned. Outcomes are sparse, so many zeros are expected, especially for purchases.

In [ ]:
group_summary = (df.groupby("segment", observed=True)
    .agg(customers=("segment", "size"), visit_rate=("visit", "mean"),
         purchase_rate=("conversion", "mean"), spend_per_customer=("spend", "mean"))
    .sort_values("customers", ascending=False))
display(group_summary.style.format({"visit_rate": "{:.2%}", "purchase_rate": "{:.2%}", "spend_per_customer": "${:.2f}"}))


## 5. Inspect pre-treatment balance

Randomization should balance characteristics measured before treatment. Standardized mean differences are descriptive checks; small differences are expected from random variation and should not be used to remove customers.

In [ ]:
def standardized_mean_difference(a, b):
    pooled_sd = ((a.var(ddof=1) + b.var(ddof=1)) / 2) ** 0.5
    return (a.mean() - b.mean()) / pooled_sd if pooled_sd else 0.0

control = df[df.segment == "No E-Mail"]
balance_rows = []
for arm in ["Mens E-Mail", "Womens E-Mail"]:
    treated = df[df.segment == arm]
    for feature in ["recency", "history", "mens", "womens", "newbie"]:
        balance_rows.append({"campaign": arm, "feature": feature,
                             "standardized_mean_difference": standardized_mean_difference(treated[feature], control[feature])})
balance = pd.DataFrame(balance_rows)
display(balance.pivot(index="feature", columns="campaign", values="standardized_mean_difference").round(3))


## Audit takeaways

- Keep all rows unless there is evidence of erroneous duplication; no customer ID is available here.
- Missing values and consistency issues should be reported before modeling.
- Only customer information measured before assignment can be used as model features.
- `segment` identifies assignment; `visit`, `conversion`, and `spend` are post-treatment outcomes.